# SmartShop — Study Results Analysis (Draft)

Draft Results-chapter notebook for the evaluation study.

**Inputs (see `../data/README.txt`):**
- Admin experiment CSV (`experiment_export.csv` or sample)
- One Google Form export — 4 sections (`form_survey.csv`):
  1. Participant ID
  2. SUS (10 official items)
  3. UEQ-S / UEQ-8 (8 official pairs)
  4. AWEQ (7 items) + 3 optional
- Optional task outcomes (`task_outcomes.csv`)

**Aligned with:** `STUDY_EVALUATION_FORMS.txt`.

**Status:** Draft — runs on sample data until you drop real exports into `../data/`.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("..").resolve()
DATA = ROOT / "data"
REPORTS = ROOT / "reports"
FIGS = REPORTS / "figures"
REPORTS.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("DATA   :", DATA)
print("REPORTS:", REPORTS)

In [ ]:
def pick_file(*candidates):
    """Prefer real export; fall back to sample for draft runs."""
    for name in candidates:
        p = DATA / name
        if p.exists():
            return p
    return None


def norm_mood(x) -> str:
    if pd.isna(x) or str(x).strip() == "":
        return ""
    return str(x).strip().lower()


def yes_rate(series: pd.Series) -> float:
    s = series.astype(str).str.strip().str.lower()
    valid = s[s.isin(["yes", "no", "true", "false", "1", "0"])]
    if valid.empty:
        return float("nan")
    return float(valid.isin(["yes", "true", "1"]).mean())


def shannon_entropy(counts: pd.Series) -> float:
    p = counts.astype(float)
    p = p[p > 0] / p.sum()
    if p.empty:
        return float("nan")
    return float(-(p * np.log2(p)).sum())


def save_table(df: pd.DataFrame, name: str) -> Path:
    out = REPORTS / name
    df.to_csv(out, index=True)
    print("Wrote", out)
    return out


def join_key_frame(left: pd.DataFrame, right: pd.DataFrame):
    """Join form → experiment on participant_id ↔ user_id (or shared participant_id)."""
    if "participant_id" in left.columns and "participant_id" in right.columns:
        return left.merge(right, on="participant_id", how="left", suffixes=("", "_form"))
    if "user_id" in left.columns and "participant_id" in right.columns:
        r = right.rename(columns={"participant_id": "user_id"})
        return left.merge(r, on="user_id", how="left", suffixes=("", "_form"))
    if "email" in left.columns and "email" in right.columns:
        return left.merge(right, on="email", how="left", suffixes=("", "_form"))
    print("No shared join key (participant_id/user_id/email).")
    return left

## 1. Load experiment CSV

Looks for `experiment_export.csv` first; otherwise uses `sample_experiment_export.csv`.

In [ ]:
exp_path = pick_file("experiment_export.csv", "sample_experiment_export.csv")
assert exp_path is not None, f"No experiment CSV found under {DATA}"

df = pd.read_csv(exp_path)
df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]

USING_SAMPLE = exp_path.name.startswith("sample_")
print(f"Loaded: {exp_path.name}  |  n={len(df)}  |  sample={USING_SAMPLE}")
df.head()

## 2. Participant / context overview

In [ ]:
overview = {
    "n_participants": len(df),
    "devices": df["device"].value_counts(dropna=False).to_dict() if "device" in df.columns else {},
    "survey_personas": df["survey_persona"].value_counts(dropna=False).to_dict()
    if "survey_persona" in df.columns
    else {},
    "guideline_personas": df["guideline_persona"].value_counts(dropna=False).to_dict()
    if "guideline_persona" in df.columns
    else {},
    "self_moods": df["self_reported_mood"].value_counts(dropna=False).to_dict()
    if "self_reported_mood" in df.columns
    else {},
}
print(json.dumps(overview, indent=2, default=str))

if {"device", "guideline_persona"}.issubset(df.columns):
    ct = pd.crosstab(df["guideline_persona"], df["device"], margins=True)
    display(ct)
    save_table(ct, "01_persona_x_device.csv")

## 3. Mood agreement

Compare `self_reported_mood` vs `model_detected_mood` (case-insensitive).

In [ ]:
CONFIDENCE_THRESHOLD = None  # e.g. 0.5

mood = df.copy()
mood["self_n"] = mood["self_reported_mood"].map(norm_mood) if "self_reported_mood" in mood.columns else ""
mood["det_n"] = mood["model_detected_mood"].map(norm_mood) if "model_detected_mood" in mood.columns else ""

has_both = (mood["self_n"] != "") & (mood["det_n"] != "")
mood_eval = mood.loc[has_both].copy()

if CONFIDENCE_THRESHOLD is not None and "model_confidence" in mood_eval.columns:
    conf = pd.to_numeric(mood_eval["model_confidence"], errors="coerce")
    mood_eval = mood_eval.loc[conf >= CONFIDENCE_THRESHOLD].copy()

mood_eval["agree"] = mood_eval["self_n"] == mood_eval["det_n"]
agreement_rate = float(mood_eval["agree"].mean()) if len(mood_eval) else float("nan")

print(f"Mood agreement rows: {len(mood_eval)}")
print(f"Agreement rate: {agreement_rate:.1%}" if not math.isnan(agreement_rate) else "Agreement rate: n/a")

confusion = pd.crosstab(
    mood_eval["self_n"].rename("self"),
    mood_eval["det_n"].rename("detected"),
    margins=True,
)
display(confusion)
save_table(confusion, "02_mood_confusion.csv")
save_table(
    pd.DataFrame(
        [{"n_compared": len(mood_eval), "n_agree": int(mood_eval["agree"].sum()) if len(mood_eval) else 0,
          "agreement_rate": agreement_rate, "confidence_threshold": CONFIDENCE_THRESHOLD}]
    ),
    "02_mood_agreement_summary.csv",
)

In [ ]:
if len(mood_eval):
    fig, ax = plt.subplots(figsize=(6, 4))
    counts = mood_eval["agree"].value_counts().reindex([True, False], fill_value=0)
    ax.bar(["Agree", "Disagree"], counts.values, color=["#2a9d8f", "#e76f51"])
    ax.set_title("Self vs model mood agreement")
    ax.set_ylabel("Participants")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.05, str(int(v)), ha="center")
    fig.tight_layout()
    fig.savefig(FIGS / "mood_agreement.png", dpi=150)
    plt.show()

## 4. Fallback rates

Uses admin flags derived from `guidelines_pipeline`.

In [ ]:
FALLBACK_COLS = [
    "used_mood_fallback",
    "used_device_fallback",
    "used_persona_fallback",
    "used_global_fill",
]
present = [c for c in FALLBACK_COLS if c in df.columns]
fallback_rates = {c: yes_rate(df[c]) for c in present}

if present:
    hard = [c for c in ["used_mood_fallback", "used_device_fallback", "used_persona_fallback"] if c in df.columns]

    def any_hard_fallback(row):
        return any(str(row.get(c, "")).strip().lower() in {"yes", "true", "1"} for c in hard)

    df["_any_hard_fallback"] = df.apply(any_hard_fallback, axis=1)
    exact_cell_rate = 1.0 - float(df["_any_hard_fallback"].mean()) if len(df) else float("nan")
else:
    exact_cell_rate = float("nan")

fallback_summary = pd.DataFrame(
    [
        {"metric": "exact_cell_rate (no mood/device/persona fallback)", "rate": exact_cell_rate},
        *[{ "metric": k, "rate": v} for k, v in fallback_rates.items()],
    ]
)
display(fallback_summary)
save_table(fallback_summary.set_index("metric"), "03_fallback_rates.csv")

if "guidelines_pipeline" in df.columns:
    display(df["guidelines_pipeline"].fillna("(blank)").value_counts().to_frame("n"))

In [ ]:
if present:
    fig, ax = plt.subplots(figsize=(7, 4))
    labels = [c.replace("used_", "") for c in present]
    vals = [fallback_rates[c] * 100 for c in present]
    ax.barh(labels, vals, color="#264653")
    ax.set_xlabel("% of participants")
    ax.set_xlim(0, 100)
    ax.set_title("Fallback usage rates")
    for i, v in enumerate(vals):
        ax.text(v + 1, i, f"{v:.0f}%", va="center")
    fig.tight_layout()
    fig.savefig(FIGS / "fallback_rates.png", dpi=150)
    plt.show()

## 5. Config diversity

Shows the adapted UI is not collapsing everyone to one layout.

In [ ]:
UI_CANDIDATES = [
    "ui_color_theme_pref",
    "ui_checkout_style",
    "ui_recommendation_type",
    "ui_desktop_navigation",
    "ui_mobile_navigation",
    "ui_desktop_product_card",
    "ui_mobile_product_card",
    "ui_desktop_grid_pref",
    "ui_mobile_grid_pref",
]
ui_cols = [c for c in UI_CANDIDATES if c in df.columns]
print("UI columns found:", ui_cols)

diversity_rows = []
for c in ui_cols:
    vc = df[c].fillna("(blank)").astype(str).value_counts()
    top_share = float(vc.iloc[0] / vc.sum()) if len(vc) else float("nan")
    diversity_rows.append(
        {
            "column": c,
            "n_unique": int(vc.shape[0]),
            "top_value": vc.index[0] if len(vc) else "",
            "top_share": top_share,
            "entropy_bits": shannon_entropy(vc),
        }
    )

diversity = pd.DataFrame(diversity_rows)
display(diversity)
save_table(diversity.set_index("column"), "04_config_diversity.csv")

combo_parts = [c for c in [
    "ui_color_theme_pref", "ui_desktop_navigation", "ui_mobile_navigation",
    "ui_desktop_product_card", "ui_mobile_product_card",
] if c in df.columns]

if combo_parts:
    combo = df[combo_parts].fillna("(blank)").astype(str).agg(" | ".join, axis=1)
    top_combos = combo.value_counts().head(10).to_frame("n")
    top_combos["share"] = top_combos["n"] / len(df)
    display(top_combos)
    save_table(top_combos, "04_top_ui_combinations.csv")
    print(f"Unique combinations: {combo.nunique()}")

In [ ]:
if "ui_color_theme_pref" in df.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    vc = df["ui_color_theme_pref"].fillna("(blank)").value_counts()
    ax.bar(vc.index.astype(str), vc.values, color="#457b9d")
    ax.set_title("Theme distribution")
    ax.set_ylabel("Participants")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(FIGS / "theme_distribution.png", dpi=150)
    plt.show()

## 6. Load Google Form (one form, 4 sections)

Expected columns after rename: `participant_id`, `sus_1…sus_10`, `ueq_1…ueq_8`,
`aweq_1…aweq_7`, and optional open text:
- `opt_1` liked most · `opt_2` what to improve · `opt_3` one aspect to improve

If Google Forms used long question text as headers, map them in `SURVEY_COLUMN_MAP`.

In [ ]:
# Map Google Forms export headers → short names (fill after your real export)
SURVEY_COLUMN_MAP = {
    # "Participant ID": "participant_id",
    # "I think that I would like to use this system frequently.": "sus_1",
    # "obstructive / supportive": "ueq_1",
    # "The website interface felt tailored to my preferences and shopping style.": "aweq_1",
    # "What did you like most about the adaptive website?": "opt_1",
    # "What would you improve?": "opt_2",
    # "If you could improve one aspect of the adaptive website, what would it be?": "opt_3",
}

survey = None
survey_path = pick_file("form_survey.csv", "sample_form_survey.csv")
if survey_path is None:
    print("No form_survey.csv found — skipping Sections 7–9 (SUS / UEQ / AWEQ).")
else:
    survey = pd.read_csv(survey_path)
    survey.columns = [c.replace("\ufeff", "").strip() for c in survey.columns]
    if SURVEY_COLUMN_MAP:
        survey = survey.rename(columns=SURVEY_COLUMN_MAP)
    print(f"Loaded survey: {survey_path.name}  n={len(survey)}")
    print("Columns:", list(survey.columns))
    display(survey.head())

## 7. SUS (official Brooke scoring → 0–100)

- Odd items (1,3,5,7,9): contribution = response − 1
- Even items (2,4,6,8,10): contribution = 5 − response
- Sum × 2.5 → 0–100. Individual items are not interpreted alone.

In [ ]:
sus_mean = float("nan")


def score_sus_row(row, cols):
    contrib = []
    for i, c in enumerate(cols, start=1):
        r = row[c]
        if pd.isna(r):
            return np.nan
        r = float(r)
        contrib.append((r - 1) if i % 2 == 1 else (5 - r))
    return sum(contrib) * 2.5


if survey is None:
    print("Skipped SUS.")
else:
    sus_cols = [f"sus_{i}" for i in range(1, 11)]
    missing = [c for c in sus_cols if c not in survey.columns]
    if missing:
        print("Missing SUS columns:", missing)
    else:
        for c in sus_cols:
            survey[c] = pd.to_numeric(survey[c], errors="coerce")
        survey["sus_score"] = survey.apply(lambda r: score_sus_row(r, sus_cols), axis=1)
        sus_mean = float(survey["sus_score"].mean())

        sus_summary = pd.DataFrame(
            [{
                "n": int(survey["sus_score"].notna().sum()),
                "mean": sus_mean,
                "sd": float(survey["sus_score"].std()),
                "median": float(survey["sus_score"].median()),
                "min": float(survey["sus_score"].min()),
                "max": float(survey["sus_score"].max()),
            }]
        )
        display(sus_summary)
        save_table(sus_summary, "05_sus_summary.csv")

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(survey["sus_score"].dropna(), bins=8, color="#1d3557", edgecolor="white")
        ax.axvline(68, color="#e9c46a", linestyle="--", label="~68 average benchmark")
        ax.axvline(sus_mean, color="#2a9d8f", linestyle="-", label=f"mean={sus_mean:.1f}")
        ax.set_xlabel("SUS score (0–100)")
        ax.set_ylabel("Participants")
        ax.set_title("SUS distribution")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGS / "sus_distribution.png", dpi=150)
        plt.show()

## 8. UEQ Short (UEQ-S / UEQ-8)

Official pairs (left → right on a 7-point scale):
1 obstructive–supportive · 2 complicated–easy · 3 inefficient–efficient · 4 confusing–clear  
5 boring–exciting · 6 not interesting–interesting · 7 conventional–inventive · 8 usual–leading edge

**Scoring:** code 1…7, transform to −3…+3 (`response − 4`).  
**Pragmatic Quality (PQ)** = mean items 1–4 · **Hedonic Quality (HQ)** = mean items 5–8.

In [ ]:
ueq_pq_mean = float("nan")
ueq_hq_mean = float("nan")
ueq_overall_mean = float("nan")

UEQ_LABELS = {
    "ueq_1": "obstructive–supportive",
    "ueq_2": "complicated–easy",
    "ueq_3": "inefficient–efficient",
    "ueq_4": "confusing–clear",
    "ueq_5": "boring–exciting",
    "ueq_6": "not interesting–interesting",
    "ueq_7": "conventional–inventive",
    "ueq_8": "usual–leading edge",
}

if survey is None:
    print("Skipped UEQ-S.")
else:
    ueq_cols = [f"ueq_{i}" for i in range(1, 9)]
    missing = [c for c in ueq_cols if c not in survey.columns]
    if missing:
        print("Missing UEQ columns:", missing)
    else:
        for c in ueq_cols:
            survey[c] = pd.to_numeric(survey[c], errors="coerce")
            # If already −3…+3, leave; if 1…7, shift to −3…+3
            if survey[c].dropna().between(1, 7).all():
                survey[c + "_t"] = survey[c] - 4
            else:
                survey[c + "_t"] = survey[c]

        t_cols = [c + "_t" for c in ueq_cols]
        pq_cols = t_cols[:4]
        hq_cols = t_cols[4:]
        survey["ueq_pq"] = survey[pq_cols].mean(axis=1)
        survey["ueq_hq"] = survey[hq_cols].mean(axis=1)
        survey["ueq_overall"] = survey[t_cols].mean(axis=1)

        ueq_pq_mean = float(survey["ueq_pq"].mean())
        ueq_hq_mean = float(survey["ueq_hq"].mean())
        ueq_overall_mean = float(survey["ueq_overall"].mean())

        item_means = pd.DataFrame({
            "item": ueq_cols,
            "pair": [UEQ_LABELS[c] for c in ueq_cols],
            "mean_-3_to_+3": [float(survey[c + "_t"].mean()) for c in ueq_cols],
            "scale": ["PQ"] * 4 + ["HQ"] * 4,
        })
        display(item_means)

        ueq_summary = pd.DataFrame([
            {"scale": "Pragmatic Quality (PQ)", "mean": ueq_pq_mean, "sd": float(survey["ueq_pq"].std())},
            {"scale": "Hedonic Quality (HQ)", "mean": ueq_hq_mean, "sd": float(survey["ueq_hq"].std())},
            {"scale": "Overall (all 8)", "mean": ueq_overall_mean, "sd": float(survey["ueq_overall"].std())},
        ])
        display(ueq_summary)
        save_table(item_means.set_index("item"), "06_ueq_item_means.csv")
        save_table(ueq_summary.set_index("scale"), "06_ueq_scale_summary.csv")

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(["PQ", "HQ", "Overall"], [ueq_pq_mean, ueq_hq_mean, ueq_overall_mean], color=["#264653", "#2a9d8f", "#e9c46a"])
        ax.axhline(0, color="#888", linewidth=0.8)
        ax.set_ylim(-3, 3)
        ax.set_ylabel("Mean (−3 … +3)")
        ax.set_title("UEQ-S scale means")
        fig.tight_layout()
        fig.savefig(FIGS / "ueq_scales.png", dpi=150)
        plt.show()

## 9. AWEQ (Adaptive Website Experience Questionnaire) — 7 items

Scale 1–5. Overall = mean of AWEQ-1…7. Also report per-item means.

In [ ]:
aweq_overall_mean = float("nan")

AWEQ_LABELS = {
    "aweq_1": "Perceived personalization",
    "aweq_2": "Adaptation appropriateness",
    "aweq_3": "Adaptation quality (natural)",
    "aweq_4": "Device adaptation",
    "aweq_5": "Emotion adaptation",
    "aweq_6": "Effectiveness (browse/find)",
    "aweq_7": "Overall acceptance vs non-adaptive",
}

if survey is None:
    print("Skipped AWEQ.")
else:
    aweq_cols = [f"aweq_{i}" for i in range(1, 8)]
    missing = [c for c in aweq_cols if c not in survey.columns]
    if missing:
        print("Missing AWEQ columns:", missing)
    else:
        for c in aweq_cols:
            survey[c] = pd.to_numeric(survey[c], errors="coerce")
        survey["aweq_mean"] = survey[aweq_cols].mean(axis=1)
        aweq_overall_mean = float(survey["aweq_mean"].mean())

        item_means = pd.DataFrame({
            "item": aweq_cols,
            "construct": [AWEQ_LABELS[c] for c in aweq_cols],
            "mean": [float(survey[c].mean()) for c in aweq_cols],
            "sd": [float(survey[c].std()) for c in aweq_cols],
        })
        display(item_means)
        print(f"Overall AWEQ mean (1–5): {aweq_overall_mean:.2f}  (sd={survey['aweq_mean'].std():.2f})")
        save_table(item_means.set_index("item"), "07_aweq_item_means.csv")

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh([AWEQ_LABELS[c] for c in aweq_cols][::-1], item_means["mean"].values[::-1], color="#457b9d")
        ax.set_xlim(1, 5)
        ax.set_xlabel("Mean (1–5)")
        ax.set_title("AWEQ item means")
        fig.tight_layout()
        fig.savefig(FIGS / "aweq_items.png", dpi=150)
        plt.show()

        # Optional open responses (qualitative)
        OPT_LABELS = {
            "opt_1": "What did you like most about the adaptive website?",
            "opt_2": "What would you improve?",
            "opt_3": "If you could improve one aspect of the adaptive website, what would it be?",
        }
        opt_cols = [c for c in ["opt_1", "opt_2", "opt_3"] if c in survey.columns]
        if opt_cols:
            print("\nOptional open-text responses (non-empty):")
            for c in opt_cols:
                vals = survey[c].dropna().astype(str).str.strip()
                vals = vals[vals != ""]
                print(f"  {c} — {OPT_LABELS.get(c, c)}: {len(vals)} responses")
                for v in vals.head(10):
                    print(f"    • {v}")
            keep = [c for c in ["participant_id"] if c in survey.columns] + opt_cols
            opt_out = survey[keep].copy()
            save_table(
                opt_out.set_index("participant_id") if "participant_id" in opt_out.columns else opt_out,
                "07_optional_open_text.csv",
            )

## 10. Join survey scores to experiment CSV

Join key: `participant_id` (form) ↔ `user_id` (admin).

In [ ]:
merged = df.copy()
if survey is not None:
    keep = [c for c in ["participant_id", "sus_score", "ueq_pq", "ueq_hq", "ueq_overall", "aweq_mean"] if c in survey.columns]
    merged = join_key_frame(df, survey[keep])
    scored = [c for c in ["sus_score", "ueq_pq", "ueq_hq", "aweq_mean"] if c in merged.columns]
    if scored:
        print("Joined score coverage:")
        for c in scored:
            print(f"  {c}: {merged[c].notna().sum()} / {len(merged)}")
        if "guideline_persona" in merged.columns and "aweq_mean" in merged.columns:
            by_p = merged.groupby("guideline_persona")[["sus_score", "ueq_pq", "ueq_hq", "aweq_mean"]].agg(["count", "mean"])
            display(by_p)
            save_table(by_p, "08_scores_by_persona.csv")
else:
    print("No survey to join.")

## 11. Task outcomes (cart / order)

Join `task_outcomes.csv` by `participant_id` (or email).

In [ ]:
task_path = pick_file("task_outcomes.csv", "sample_task_outcomes.csv")
if task_path is None:
    print("No task outcomes file — skipping.")
else:
    tasks = pd.read_csv(task_path)
    tasks.columns = [c.replace("\ufeff", "").strip() for c in tasks.columns]
    print(f"Loaded tasks: {task_path.name}  n={len(tasks)}")
    merged_t = join_key_frame(merged, tasks)
    for col in ["reached_cart", "placed_order"]:
        if col in merged_t.columns:
            rate = yes_rate(merged_t[col])
            print(f"{col}: {rate:.1%}" if not math.isnan(rate) else f"{col}: n/a")
    task_summary = pd.DataFrame([{
        "n_joined": int(merged_t["reached_cart"].notna().sum()) if "reached_cart" in merged_t.columns else 0,
        "reached_cart_rate": yes_rate(merged_t["reached_cart"]) if "reached_cart" in merged_t.columns else np.nan,
        "placed_order_rate": yes_rate(merged_t["placed_order"]) if "placed_order" in merged_t.columns else np.nan,
    }])
    display(task_summary)
    save_table(task_summary, "09_task_outcomes.csv")
    merged = merged_t

## 12. Thesis-ready results summary

In [ ]:
summary_rows = [
    {"section": "Sample", "metric": "N participants (admin)", "value": len(df)},
    {"section": "Sample", "metric": "Using sample data?", "value": USING_SAMPLE},
    {"section": "Mood", "metric": "Agreement rate", "value": round(agreement_rate, 3) if not math.isnan(agreement_rate) else None},
    {"section": "Fallbacks", "metric": "Exact-cell rate", "value": round(exact_cell_rate, 3) if not math.isnan(exact_cell_rate) else None},
]
for k, v in fallback_rates.items():
    summary_rows.append({"section": "Fallbacks", "metric": k, "value": round(v, 3) if not math.isnan(v) else None})

if "ui_color_theme_pref" in df.columns:
    summary_rows.append({"section": "Diversity", "metric": "Unique themes",
                         "value": int(df["ui_color_theme_pref"].nunique(dropna=True))})

if not math.isnan(sus_mean):
    summary_rows.append({"section": "SUS", "metric": "Mean SUS (0–100)", "value": round(sus_mean, 1)})
if not math.isnan(ueq_pq_mean):
    summary_rows.append({"section": "UEQ-S", "metric": "Mean PQ (−3…+3)", "value": round(ueq_pq_mean, 2)})
    summary_rows.append({"section": "UEQ-S", "metric": "Mean HQ (−3…+3)", "value": round(ueq_hq_mean, 2)})
    summary_rows.append({"section": "UEQ-S", "metric": "Mean overall (−3…+3)", "value": round(ueq_overall_mean, 2)})
if not math.isnan(aweq_overall_mean):
    summary_rows.append({"section": "AWEQ", "metric": "Mean AWEQ (1–5)", "value": round(aweq_overall_mean, 2)})

results_summary = pd.DataFrame(summary_rows)
display(results_summary)
save_table(results_summary, "00_results_summary.csv")

print("\nReports:", REPORTS)
print("Figures:", FIGS)
if USING_SAMPLE:
    print("\n⚠️  Still on SAMPLE data. Drop real exports into data/ and re-run for thesis numbers.")

## Notes for the thesis

- **SUS:** report mean (0–100); do not interpret single items.
- **UEQ-S:** report PQ and HQ (and optional overall) on −3…+3.
- **AWEQ:** report overall mean + per-item (personalization, device, emotion, acceptance).
- Cite **matrix coverage** when discussing fallbacks (`MATRIX_COVERAGE_APPENDIX.txt`).
- Join everything with the same **Participant ID** as SmartShop `user_id`.

**Next:** export real form → `data/form_survey.csv`, map headers in `SURVEY_COLUMN_MAP` if needed, Run All.